## ✂️ PASO 6 — Partición estratificada y balanceo de clases

In [ ]:
# ── Vectores de entrada y salida ──────────────────────────────
X = df[FEATURES].values.astype(np.float32)
y = df['target'].values

# ── Partición estratificada 70 / 15 / 15 ──────────────────────
X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.1765, stratify=y_tmp, random_state=SEED)

print('=== Partición del dataset ===')
for name, xs, ys in [('Entrenamiento', X_train, y_train),
                      ('Validación',    X_val,   y_val),
                      ('Prueba',        X_test,  y_test)]:
    n0 = (ys==0).sum(); n1 = (ys==1).sum()
    print(f'  {name:<14}: {len(xs):6,} registros | '
          f'No readmit.: {n0:,} ({n0/len(ys)*100:.1f} %) | '
          f'Readmitido: {n1:,} ({n1/len(ys)*100:.1f} %)')

# ── Normalización (StandardScaler ajustado solo en train) ─────
sc      = StandardScaler()
Xtr_s   = sc.fit_transform(X_train).astype(np.float32)
Xv_s    = sc.transform(X_val).astype(np.float32)
Xte_s   = sc.transform(X_test).astype(np.float32)

# ── Tres estrategias de balanceo ──────────────────────────────
sm  = SMOTE(random_state=SEED)
ros = RandomOverSampler(random_state=SEED)
rus = RandomUnderSampler(random_state=SEED)

Xsm,  ysm  = sm.fit_resample(Xtr_s, y_train)
Xros, yros = ros.fit_resample(Xtr_s, y_train)
Xrus, yrus = rus.fit_resample(Xtr_s, y_train)

Xsm  = Xsm.astype(np.float32);  ysm  = ysm.astype(np.float32)
Xros = Xros.astype(np.float32); yros = yros.astype(np.float32)
Xrus = Xrus.astype(np.float32); yrus = yrus.astype(np.float32)

print('\n=== Conjuntos de entrenamiento tras balanceo ===')
for name, xs, ys in [('Sin balanceo', Xtr_s, y_train),
                      ('SMOTE',        Xsm,   ysm),
                      ('ROS',          Xros,  yros),
                      ('RUS',          Xrus,  yrus)]:
    u, c = np.unique(ys, return_counts=True)
    print(f'  {name:<15}: {len(xs):6,} muestras | dist: {dict(zip(u.astype(int), c))}')

print('\n✓ Partición y balanceo completados.')

## 🤖 PASO 7 — Modelos de Machine Learning Clásico
Se entrenan Regresión Logística, Random Forest y SVM con SMOTE.

In [ ]:
print('Entrenando modelos de ML clásico con SMOTE...')

# ── Entrenamiento ─────────────────────────────────────────────
lr  = LogisticRegression(max_iter=1000, random_state=SEED, C=1.0)
rf  = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
svm = SVC(kernel='rbf', probability=True, random_state=SEED, max_iter=2000)

lr.fit(Xsm, ysm);       print('  ✓ Regresión Logística entrenada.')
rf.fit(Xsm, ysm);       print('  ✓ Random Forest entrenado.')
svm.fit(Xsm[:5000], ysm[:5000]);  print('  ✓ SVM entrenado (submuestra 5 000 por velocidad).')

# ── Evaluación en conjunto de prueba ──────────────────────────
print('\n=== Métricas en el conjunto de prueba ===')
resultados_ml = {}
for nombre, modelo in [('Reg. Logística', lr), ('Random Forest', rf), ('SVM (RBF)', svm)]:
    probs = modelo.predict_proba(Xte_s)[:, 1]
    preds = (probs > 0.5).astype(int)
    resultados_ml[nombre] = {
        'AUC-ROC':   round(roc_auc_score(y_test, probs), 4),
        'F1-macro':  round(f1_score(y_test, preds, average='macro'), 4),
        'Precisión': round(precision_score(y_test, preds), 4),
        'Recall':    round(recall_score(y_test, preds), 4),
        'Accuracy':  round((preds == y_test).mean(), 4),
    }
    print(f'\n  {nombre}:')
    for k, v in resultados_ml[nombre].items():
        print(f'    {k:<12}: {v}')

print('\n✓ Modelos de ML evaluados.')